# 12 - ExecutionPlan and Strategy Selection

## Objetivo
Mostrar como `IntentPlanner` transforma o mesmo intent em planos diferentes
conforme a `RoutingStrategy` escolhida, usando uma topologia em diamante
onde as estratégias **realmente divergem** - e confirmar por execução real
que a rota aceita pelo SeQUeNCe corresponde à rota aplicada pelo planner.

## Conceitos
- Topologia em diamante (`ibqn.demos.topologies.diamond_spec`):

```text
        bad (alta atenuação, rápido a degradar)
       /   \
     r1     r3
       \   /
     good1-good2 (baixa atenuação, mais saltos)
```

- `ShortestHopCountRouting`: menos saltos, ignora perda/fidelidade.
- `LeastLossRouting`: minimiza perda óptica acumulada (`distance_m x
  attenuation_db_per_m` somado por salto).
- `HighestFidelityRouting`: maximiza a fidelidade fim a fim estimada
  (`planning.feasibility.estimate_swap_only_fidelity`), antes de qualquer
  purificação.
- O planner **controla**: qual rota candidata usar, se purificação é
  necessária (estimativa), quantas memórias reservar por nó. O SeQUeNCe
  **decide sozinho**: ordem de swap (bisseção binária do caminho), sucesso/
  falha de cada tentativa física, e o número real de rounds de purificação
  em tempo de execução.


## Configuração


In [1]:
from ibqn.demos.topologies import diamond_spec
from ibqn.demos.intents import diamond_intent
from ibqn.network.capabilities import NetworkCapabilities
from ibqn.planning.routing import ShortestHopCountRouting, LeastLossRouting, HighestFidelityRouting

SEED = 0
spec = diamond_spec()
intent = diamond_intent(requested_pairs=10, min_fidelity=0.6)
capabilities = NetworkCapabilities(spec)

strategies = {
    "ShortestHopCount": ShortestHopCountRouting(),
    "LeastLoss": LeastLossRouting(),
    "HighestFidelity": HighestFidelityRouting(),
}
print(f"intent: {intent.endpoints.source} -> {intent.endpoints.destination}, min_fidelity={intent.requirements.min_fidelity}")


2026-07-13 07:28:31,724 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'requested_pairs' -> 'reserved_memory_slots' (see docs/intent_resource_semantics.md)


2026-07-13 07:28:31,728 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'start_time' -> 'start_time_s' (see docs/intent_resource_semantics.md)


2026-07-13 07:28:31,729 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'duration' -> 'duration_s' (see docs/intent_resource_semantics.md)


intent: r1 -> r3, min_fidelity=0.6


## Execução (estimativas do planner)

Todos os caminhos candidatos que cada estratégia considera - não apenas o
vencedor (`rank=0`) - avaliados pela mesma função que `IntentPlanner` usa
internamente (`planning.feasibility.evaluate_route`).


In [2]:
from ibqn.demos.planning import compare_routing_strategies

estimates_df = compare_routing_strategies(capabilities, intent, strategies)
estimates_df


,strategy,rank,route,hops,estimated_loss_db,estimated_fidelity,purification_required,feasible
0,ShortestHopCount,0,r1 -> bad -> r3,2,40.000,0.6864,False,True
1,ShortestHopCount,1,r1 -> good1 -> good2 -> r3,3,0.015,0.6047,True,True
2,LeastLoss,0,r1 -> good1 -> good2 -> r3,3,0.015,0.6047,True,True
3,LeastLoss,1,r1 -> bad -> r3,2,40.000,0.6864,False,True
4,HighestFidelity,0,r1 -> bad -> r3,2,40.000,0.6864,False,True
5,HighestFidelity,1,r1 -> good1 -> good2 -> r3,3,0.015,0.6047,True,True


In [3]:
winners = estimates_df[estimates_df["rank"] == 0].set_index("strategy")["route"]
print("Winning route per strategy:")
for name, route in winners.items():
    print(f"  {name}: {route}")

assert winners["ShortestHopCount"] == winners["HighestFidelity"] == "r1 -> bad -> r3"
assert winners["LeastLoss"] == "r1 -> good1 -> good2 -> r3"
print("\nConfirmed: LeastLossRouting genuinely disagrees with the other two strategies on this topology.")


Winning route per strategy:
  ShortestHopCount: r1 -> bad -> r3
  LeastLoss: r1 -> good1 -> good2 -> r3
  HighestFidelity: r1 -> bad -> r3

Confirmed: LeastLossRouting genuinely disagrees with the other two strategies on this topology.


## Resultados (execução real)

Rodando cada estratégia de fato (mesma seed, mesmo intent) e confirmando
que a reserva aceita pelo SeQUeNCe usou exatamente a rota que o plano
determinou.


In [4]:
from sequence.utils import metrics
from ibqn.network.sequence_adapter import SequenceAdapter
from ibqn.execution.sequence_executor import SequenceExecutor
from ibqn.intent.repository import IntentRepository
from ibqn.planning.planner import IntentPlanner

def deploy_and_confirm_route(strategy, seed):
    metrics.configure()  # fresh storage: this notebook runs 3 independent simulations in the same kernel
    plan = IntentPlanner(capabilities, routing_strategy=strategy).plan(intent)
    adapter = SequenceAdapter(spec, seed=seed)
    repository = IntentRepository()
    executor = SequenceExecutor(adapter, repository)
    executor.deploy(intent, plan)
    executor.run()

    source_router = adapter.get_router(intent.endpoints.source)
    reservation = source_router.network_manager.protocol_stack[-1].accepted_reservations[0]
    assert reservation.path == plan.route, (
        f"planned route {plan.route} does not match the accepted reservation's real path {reservation.path}"
    )
    return plan.route


for name, strategy in strategies.items():
    route = deploy_and_confirm_route(strategy, SEED)
    print(f"{name}: planned route confirmed accepted by SeQUeNCe -> {' -> '.join(route)}")


2026-07-13 07:28:31,816 INFO     ibqn.ibqn.planning.planner intent_id=intent-diamond-demo - planned route r1->bad->r3, estimated_fidelity=0.6864, purification=False


2026-07-13 07:28:31,823 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - topology built: 5 routers, 5 quantum links, seed=0


2026-07-13 07:28:31,824 INFO     ibqn.ibqn.intent.repository intent_id=intent-diamond-demo - intent registered


2026-07-13 07:28:31,825 INFO     ibqn.ibqn.intent.repository intent_id=intent-diamond-demo - intent transitioned to VALIDATED (schema validated)


2026-07-13 07:28:31,826 INFO     ibqn.ibqn.intent.repository intent_id=intent-diamond-demo - intent transitioned to PLANNING (invoking IntentPlanner)


...

2026-07-13 07:28:40,062 INFO     ibqn.ibqn.execution.sequence_executor intent_id=intent-diamond-demo - submitting reservation r1 -> r3, reserved_memory_slots=10, min_fidelity=0.600


2026-07-13 07:28:40,062 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - running simulation, stop_time=0.200s


2026-07-13 07:28:40,066 INFO     ibqn.ibqn.intent.repository intent_id=intent-diamond-demo - intent transitioned to ACTIVE (reservation approved)


LeastLoss: planned route confirmed accepted by SeQUeNCe -> r1 -> good1 -> good2 -> r3


HighestFidelity: planned route confirmed accepted by SeQUeNCe -> r1 -> bad -> r3


## Campos controlados pelo IBQN vs. delegados ao SeQUeNCe


In [5]:
import pandas as pd

control_df = pd.DataFrame([
    {"Field": "Route (which nodes)", "Controlled by": "IBQN (RoutingStrategy)"},
    {"Field": "Purification attempted (yes/no, estimate)", "Controlled by": "IBQN (PurificationStrategy)"},
    {"Field": "Memories reserved per node", "Controlled by": "IBQN (planning.resource_allocation, mirrors RSVPProtocol.schedule)"},
    {"Field": "Reservation time window", "Controlled by": "IBQN (intent.requirements.start_time_s/duration_s)"},
    {"Field": "Swap order along the route", "Controlled by": "SeQUeNCe (ResourceManager.generate_load_rules, binary bisection)"},
    {"Field": "Per-attempt success/failure of generation/swap/purification", "Controlled by": "SeQUeNCe (real physics: photon loss, BSM success probability)"},
    {"Field": "Actual number of purification rounds executed", "Controlled by": "SeQUeNCe ('until_target' reservation mode)"},
    {"Field": "BSM node placement on a quantum link", "Controlled by": "SeQUeNCe (RouterNetTopo meet_in_the_middle)"},
])
control_df

,Field,Controlled by
0,Route (which nodes),IBQN (RoutingStrategy)
1,"Purification attempted (yes/no, estimate)",IBQN (PurificationStrategy)
2,Memories reserved per node,"IBQN (planning.resource_allocation, mirrors RS..."
3,Reservation time window,IBQN (intent.requirements.start_time_s/duratio...
4,Swap order along the route,"SeQUeNCe (ResourceManager.generate_load_rules,..."
5,Per-attempt success/failure of generation/swap...,"SeQUeNCe (real physics: photon loss, BSM succe..."
6,Actual number of purification rounds executed,SeQUeNCe ('until_target' reservation mode)
7,BSM node placement on a quantum link,SeQUeNCe (RouterNetTopo meet_in_the_middle)


## Interpretação
`LeastLossRouting` escolhe o desvio de 3 saltos (`good1`/`good2`) mesmo
tendo mais saltos que `bad`, porque sua perda óptica acumulada é muito
menor - as outras duas estratégias preferem `bad` (menos saltos /
fidelidade estimada mais alta, já que menos swaps significa menos
degradação). Em todos os três casos, a reserva realmente aceita pelo
SeQUeNCe usou exatamente a rota que o plano determinou: o planejador
decide o caminho, mas quem efetivamente reserva os recursos e negocia a
aceitação salto a salto é o `RSVPProtocol` do próprio SeQUeNCe.

## Limitações
- `HighestFidelityRouting` enumera caminhos simples via
  `networkx.all_simple_paths` - só é tratável para topologias pequenas
  (`docs/limitations.md`).
- A ordem de swap não é controlável por nenhuma estratégia desta versão
  (ver notebook 02 do H1); isso não afeta a estimativa de fidelidade do
  estimador padrão (`ConservativeMinEstimator`, associativo por
  construção) - mas o `SequenceConsistentEstimator` (Fase J1) reproduz o
  mecanismo real do SeQUeNCe, onde a ordem de swap **importa**
  genuinamente (ver `docs/fidelity_estimation_model.md`). "Estratégia de
  swapping" aqui continua sendo só descritiva (`SwappingStrategy.describe`),
  não uma decisão real.

## Conclusão
Três estratégias de roteamento comparadas numa topologia onde elas
genuinamente divergem, com confirmação real de que a rota escolhida pelo
planner é a mesma que o SeQUeNCe aceitou em cada caso.